In [ ]:
from pathlib import Path
import re
import pandas as pd
import pdfplumber

# =========================
# CONFIG
# =========================
PDF_PATH = Path(r"/home/hello/Desktop/Documents/Case_vs_BBG/Bloomberg´s evidence/(2) Leonardo Fabian Maranon Mejia v Bloomberg LP - Disclosure bundle of documents 11.12.2024_SEARCHABLE.pdf")
PAGE_START, PAGE_END = 220, 238  # 1-indexed page numbers
OUT_DIR = Path("./extract_out_utc_v3_1")
OUT_DIR.mkdir(exist_ok=True)

# =========================
# REGEX
# =========================
DATE_DMY_RE = re.compile(r"(\d{1,2}/\d{1,2}/\d{4})")           # dd/mm/yyyy (1-digit allowed)
TIME_RE = re.compile(r"\b(\d{2}:\d{2}(?::\d{2})?)\b")          # HH:MM or HH:MM:SS
MY_ANYWHERE_RE = re.compile(r"(\d{1,2})/(\d{4})")              # m/yyyy anywhere (even inside tokens)

F_TOKEN_RE = re.compile(r"\b(F\d{2,8}\d{1,2}/\d{4})\b|\b(F\d{2,8})\b")

KEYWORDS = ["LPAD", "Launchpad", "WorkCentre", "WorkCenter", "crash", "crashed", "crashes", "queue", "ticket", "ADD"]

# =========================
# HELPERS
# =========================
def norm_cell(x) -> str:
    if x is None:
        return ""
    s = str(x)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_month_year_anywhere(s: str):
    """
    Finds any m/yyyy occurrence inside the string (even inside tokens),
    keeps only valid months 1..12, returns last valid occurrence as MM/YYYY.
    """
    if not s:
        return None
    matches = MY_ANYWHERE_RE.findall(s)
    valid = [(m, y) for (m, y) in matches if 1 <= int(m) <= 12]
    if not valid:
        return None
    m, y = valid[-1]
    return f"{int(m):02d}/{y}"

def extract_full_date(s: str):
    m = DATE_DMY_RE.search(s or "")
    return m.group(1) if m else None

def extract_time(s: str):
    m = TIME_RE.search(s or "")
    return m.group(1) if m else None

def extract_f_token(s: str):
    if not s:
        return None
    m = F_TOKEN_RE.search(s)
    if not m:
        return None
    return m.group(1) or m.group(2)

# =========================
# EXTRACTION
# =========================
rows = []

with pdfplumber.open(PDF_PATH) as pdf:
    for pno in range(PAGE_START, PAGE_END + 1):
        page = pdf.pages[pno - 1]

        # ---- 1) Try tables first ----
        tables = page.extract_tables() or []
        if tables:
            for ti, table in enumerate(tables):
                for ri, r in enumerate(table):
                    cells = [norm_cell(c) for c in r]
                    joined = " | ".join([c for c in cells if c]).strip()
                    if not joined:
                        continue

                    is_relevant = any(k.lower() in joined.lower() for k in KEYWORDS)
                    date_utc = extract_full_date(joined)
                    month_year_utc = extract_month_year_anywhere(joined) if not date_utc else None
                    time_utc = extract_time(joined)
                    f_token = extract_f_token(joined)

                    rows.append({
                        "page": pno,
                        "source": f"table[{ti}]",
                        "row_idx": ri,
                        "date_utc": date_utc,
                        "month_year_utc": month_year_utc,
                        "time_utc": time_utc,
                        "f_token": f_token,
                        "relevant_flag": is_relevant,
                        "row_text": joined
                    })
            continue  # don't double-count with fallback if table extraction worked

        # ---- 2) Fallback: reconstruct lines from words ----
        words = page.extract_words() or []
        if not words:
            continue

        tol = 3  # line grouping tolerance
        words_sorted = sorted(words, key=lambda w: (round(w["top"] / tol) * tol, w["x0"]))
        current_key = None
        current_words = []

        def flush_line():
            if not current_words:
                return

            line = " ".join(w["text"] for w in current_words)
            line = re.sub(r"\s+", " ", line).strip()

            # clear in-place (no rebinding => no nonlocal needed)
            current_words.clear()

            if not line:
                return

            is_relevant = any(k.lower() in line.lower() for k in KEYWORDS)
            date_utc = extract_full_date(line)
            month_year_utc = extract_month_year_anywhere(line) if not date_utc else None
            time_utc = extract_time(line)
            f_token = extract_f_token(line)

            rows.append({
                "page": pno,
                "source": "words_line",
                "row_idx": None,
                "date_utc": date_utc,
                "month_year_utc": month_year_utc,
                "time_utc": time_utc,
                "f_token": f_token,
                "relevant_flag": is_relevant,
                "row_text": line
            })

        for w in words_sorted:
            key = round(w["top"] / tol) * tol
            if current_key is None:
                current_key = key
            if key != current_key:
                flush_line()
                current_key = key
            current_words.append(w)

        flush_line()

df = pd.DataFrame(rows)

# =========================
# OUTPUTS (Pure extraction)
# =========================
# =========================
# Build df_ts
# =========================
df_ts = df[
    df["time_utc"].notna() &
    (df["date_utc"].notna() | df["month_year_utc"].notna())
].copy()

df_ts["date_like"] = df_ts["date_utc"].fillna(df_ts["month_year_utc"])
df_ts["timestamp_utc_str"] = df_ts["date_like"] + " " + df_ts["time_utc"]

df_ts["dt_parse"] = pd.to_datetime(
    df_ts["timestamp_utc_str"],
    errors="coerce",
    dayfirst=True
)

# =========================
# Timezone enrichment (ONLY where full date exists)
# =========================
import pandas as pd
from zoneinfo import ZoneInfo

def add_pit_and_time_category(
    df_ts: pd.DataFrame,
    dt_parse_col: str = "dt_parse",
    uk_tz: str = "Europe/London",
) -> pd.DataFrame:
    """
    Enrich df_ts with:
      - PIT timestamps (UTC + UK local, DST-aware)
      - UK offset + DST flags
      - UK hour/minute + weekday/weekend
      - One clean classification column: time_category ∈ {WEEKEND, LUNCH, WORK, OUTSIDE}

    Assumptions:
      - df_ts[dt_parse_col] is a naive datetime representing a UTC clock reading.
      - Rows where dt_parse is NaT remain un-enriched (kept as NaN/NA).
    """
    out = df_ts.copy()
    UK_TZ = ZoneInfo(uk_tz)

    mask = out[dt_parse_col].notna()

    # ---- PIT (UTC) ----
    out.loc[mask, "dt_utc"] = out.loc[mask, dt_parse_col].dt.tz_localize("UTC")

    # ---- PIT (UK local) ----
    out.loc[mask, "dt_uk"] = out.loc[mask, "dt_utc"].dt.tz_convert(UK_TZ)

    # ---- Offsets / DST ----
    out.loc[mask, "uk_offset"] = out.loc[mask, "dt_uk"] - out.loc[mask, "dt_utc"]
    out.loc[mask, "uk_offset_minutes"] = (
        out.loc[mask, "uk_offset"].dt.total_seconds().div(60).astype("Int64")
    )
    out.loc[mask, "uk_offset_hours"] = out.loc[mask, "uk_offset_minutes"] / 60
    out.loc[mask, "uk_tz_abbr"] = out.loc[mask, "dt_uk"].dt.strftime("%Z")  # GMT or BST
    out.loc[mask, "uk_is_dst"] = out.loc[mask, "uk_offset_hours"] == 1

    # ---- Readable PIT strings ----
    out.loc[mask, "pit_utc_iso"] = out.loc[mask, "dt_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    out.loc[mask, "pit_uk_iso"] = out.loc[mask, "dt_uk"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
    out.loc[mask, "pit"] = out.loc[mask, "dt_uk"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")  # watch-time

    # ---- Time components (UK local) ----
    out.loc[mask, "uk_hour"] = out.loc[mask, "dt_uk"].dt.hour
    out.loc[mask, "uk_minute"] = out.loc[mask, "dt_uk"].dt.minute
    out.loc[mask, "uk_weekday"] = out.loc[mask, "dt_uk"].dt.weekday  # 0=Mon ... 6=Sun
    out.loc[mask, "is_weekend"] = out.loc[mask, "uk_weekday"] >= 5

    # ---- Single clean classification column ----
    # Priority: WEEKEND > LUNCH > WORK > OUTSIDE
    out["time_category"] = pd.NA
    out.loc[mask, "time_category"] = "OUTSIDE"
    out.loc[mask & (out["is_weekend"] == True), "time_category"] = "WEEKEND"
    out.loc[mask & (out["is_weekend"] == False) & (out["uk_hour"] >= 12) & (out["uk_hour"] < 13), "time_category"] = "LUNCH"
    out.loc[mask & (out["is_weekend"] == False) & (out["uk_hour"] >= 8) & (out["uk_hour"] < 18) & (out["time_category"] != "LUNCH"), "time_category"] = "WORK"

    return out

# Usage:
df_ts = add_pit_and_time_category(df_ts)


# =========================
# Save AFTER enrichment
# =========================
df.to_csv(OUT_DIR / "raw_rows_p220_238.csv", index=False)

df_ts.sort_values(["dt_parse", "page"], na_position="last") \
     .to_csv(OUT_DIR / "timestamps_p220_238.csv", index=False)



print("=== v3.1 extraction summary ===")
print(f"PDF: {PDF_PATH.name}")
print(f"Pages: {PAGE_START}-{PAGE_END}")
print(f"Raw rows captured: {len(df)}")
print(f"Timestamp-like rows (time + date_or_monthyear): {len(df_ts)}")
print("Saved:")
print(" -", (OUT_DIR / "raw_rows_p220_238.csv").resolve())
print(" -", (OUT_DIR / "timestamps_p220_238.csv").resolve())


=== v3.1 extraction summary ===
PDF: (2) Leonardo Fabian Maranon Mejia v Bloomberg LP - Disclosure bundle of documents 11.12.2024_SEARCHABLE.pdf
Pages: 220-238
Raw rows captured: 1088
Timestamp-like rows (time + date_or_monthyear): 886
Saved:
 - /home/hello/Projects/Statements/extract_out_utc_v3_1/raw_rows_p220_238.csv
 - /home/hello/Projects/Statements/extract_out_utc_v3_1/timestamps_p220_238.csv


/tmp/ipykernel_629842/3767889680.py:171: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_ts["dt_parse"] = pd.to_datetime(


In [5]:
print(df_ts["time_category"].value_counts())


time_category
WORK       602
WEEKEND    194
LUNCH       82
OUTSIDE      7
Name: count, dtype: int64
